In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sqlalchemy import create_engine

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score
)

from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor

pd.set_option("display.max_columns", None)

In [ ]:
username = "postgres"
password = "password"
host = "localhost"
port = "5432"
database = "anomaly_detection_db"

engine = create_engine(
    f"postgresql://{username}:{password}@{host}:{port}/{database}"
)

In [ ]:
query = "SELECT * FROM creditcard_fraud;"

df = pd.read_sql(query, engine)

df = df.drop_duplicates()

X = df.drop("Class", axis=1)
y = df["Class"]

scaler = StandardScaler()

X[["Time", "Amount"]] = scaler.fit_transform(
    X[["Time", "Amount"]]
)

print(X.shape)
print(y.shape)

# Credit Card Fraud Detection Models

This notebook focuses on anomaly detection model development for highly imbalanced fraud detection data.

The objective is to compare unsupervised anomaly detection approaches and evaluate how effectively rare fraudulent transactions can be identified.

In [ ]:
iso_forest = IsolationForest(
    n_estimators=100,
    contamination=0.0017,
    random_state=42
)

iso_forest.fit(X)

## Isolation Forest Prediction

Isolation Forest returns predictions as:

- 1 → normal observation
- -1 → anomaly

Since our fraud labels are defined as:

- 0 → normal transaction
- 1 → fraudulent transaction

the predictions were remapped to align with the business problem and evaluation metrics.

In [ ]:
y_pred = iso_forest.predict(X)

np.unique(y_pred, return_counts=True)

In [ ]:
y_pred = np.where(y_pred == -1, 1, 0)

np.unique(y_pred, return_counts=True)

In [ ]:
print(classification_report(y, y_pred))

In [ ]:
cm = confusion_matrix(y, y_pred)

print(cm)

## Isolation Forest Evaluation

Isolation Forest performs extremely well on normal transactions due to the strong class imbalance.

However, fraud detection performance remains limited, with only 16% recall for fraudulent transactions.

This indicates that while the model captures some anomaly patterns, many fraud cases remain undetected and further model comparison is necessary.

## Local Outlier Factor (LOF)

Local Outlier Factor is a distance-based anomaly detection algorithm that detects observations with significantly lower local density compared to their neighbors.

Unlike Isolation Forest, LOF focuses on neighborhood structure and local deviation patterns, which may improve fraud detection performance for rare abnormal transactions.

In [ ]:
lof = LocalOutlierFactor(
    n_neighbors=20,
    contamination=0.0017
)

y_pred_lof = lof.fit_predict(X)

np.unique(y_pred_lof, return_counts=True)

In [ ]:
y_pred_lof = np.where(y_pred_lof == -1, 1, 0)

np.unique(y_pred_lof, return_counts=True)

In [ ]:
print(classification_report(y, y_pred_lof))

In [ ]:
cm_lof = confusion_matrix(y, y_pred_lof)

print(cm_lof)

## Local Outlier Factor Evaluation

LOF failed to detect fraudulent transactions and achieved 0% recall for the fraud class.

This indicates that density-based local anomaly detection was not effective for this dataset.

Compared to LOF, Isolation Forest performed significantly better and proved to be a more suitable anomaly detection approach for credit card fraud detection.

## Model Comparison

The performance of anomaly detection models is compared based on fraud detection capability rather than overall accuracy.

Since the dataset is highly imbalanced, recall for the fraud class is considered the most important evaluation metric.

The goal is to identify the model that captures the highest number of fraudulent transactions while minimizing false alarms.

In [ ]:
comparison = pd.DataFrame({
    "Model": [
        "Isolation Forest",
        "Local Outlier Factor"
    ],
    "Fraud Recall": [
        0.16,
        0.00
    ],
    "Fraud Precision": [
        0.16,
        0.00
    ],
    "Fraud F1-Score": [
        0.16,
        0.00
    ]
})

comparison

In [ ]:
plt.figure(figsize=(8,5))
sns.barplot(
    data=comparison,
    x="Model",
    y="Fraud Recall"
)

plt.title("Model Comparison by Fraud Recall")
plt.savefig(
    "model_comparison_recall.png",
    bbox_inches="tight"
)

plt.show()

# Final Conclusion

Among the evaluated anomaly detection models, Isolation Forest demonstrated the strongest fraud detection capability for the credit card dataset.

Although overall fraud recall remains limited due to the extreme class imbalance, Isolation Forest significantly outperformed Local Outlier Factor and proved more effective in identifying rare fraudulent transactions.

This suggests that tree-based anomaly detection methods are more suitable than density-based local methods for this financial anomaly detection problem.